# Day 1 — Mutual Fund Analytics: Exploratory Data Analysis

Bluestock Fintech Capstone. Loads the 10 raw datasets, explores the fund master, and validates AMFI codes between `fund_master` and `nav_history`.

In [ ]:
import pandas as pd, numpy as np
from pathlib import Path
RAW = Path('../data/raw')
pd.set_option('display.max_columns', 40)

## 1. Load all 10 datasets

In [ ]:
files = {'fund_master':'01_fund_master.csv','nav_history':'02_nav_history.csv','aum_by_house':'03_aum_by_fund_house.csv','monthly_sip':'04_monthly_sip_inflows.csv','category_inflows':'05_category_inflows.csv','folio_count':'06_industry_folio_count.csv','performance':'07_scheme_performance.csv','transactions':'08_investor_transactions.csv','holdings':'09_portfolio_holdings.csv','benchmark':'10_benchmark_indices.csv'}
d = {k: pd.read_csv(RAW/v) for k,v in files.items()}
pd.DataFrame([(k,*v.shape) for k,v in d.items()], columns=['dataset','rows','cols'])

            dataset   rows  cols
0       fund_master     40    15
1       nav_history  46000     3
2      aum_by_house     90     5
3       monthly_sip     48     6
4  category_inflows    144     3
5       folio_count     21     6
6       performance     40    19
7      transactions  32778    13
8          holdings    322     8
9         benchmark   8050     3

## 2. Fund master — structure

In [ ]:
fm = d['fund_master']
print('Fund houses:', sorted(fm['fund_house'].unique()))
print()
print('Categories:', sorted(fm['category'].unique()))
print()
print('Sub-categories:', sorted(fm['sub_category'].unique()))
print()
print('Risk grades:', sorted(fm['risk_category'].unique()))
print()
print('SEBI category codes:', sorted(fm['sebi_category_code'].unique()))

Fund houses: ['Aditya Birla Sun Life MF', 'Axis Mutual Fund', 'DSP Mutual Fund', 'HDFC Mutual Fund', 'ICICI Prudential MF', 'Kotak Mahindra MF', 'Mirae Asset MF', 'Nippon India MF', 'SBI Mutual Fund', 'UTI Mutual Fund']

Categories: ['Debt', 'Equity']

Sub-categories: ['ELSS', 'Flexi Cap', 'Gilt', 'Index', 'Index/ETF', 'Large & Mid Cap', 'Large Cap', 'Liquid', 'Mid Cap', 'Short Duration', 'Small Cap', 'Value']

Risk grades: ['High', 'Low', 'Moderate', 'Moderately High', 'Very High']

SEBI category codes: ['DC01', 'DC02', 'EC01', 'EC02', 'EC03', 'EC04', 'EC05', 'EC06', 'EI01']


In [ ]:
codes = fm['amfi_code'].astype(str)
print('length distribution:', codes.str.len().value_counts().to_dict())
print('fully numeric:', codes.str.fullmatch(r'\d+').mean())
print('range:', codes.min(),'..',codes.max())

length distribution: {6: 40}
fully numeric: 1.0
range: 100016 .. 149324


In [ ]:
fm.groupby(['category','sub_category']).size().rename('n_schemes').reset_index()

   category     sub_category  n_schemes
0      Debt             Gilt          2
1      Debt           Liquid          3
2      Debt   Short Duration          1
3    Equity             ELSS          1
4    Equity        Flexi Cap          2
5    Equity            Index          1
6    Equity        Index/ETF          1
7    Equity  Large & Mid Cap          1
8    Equity        Large Cap         14
9    Equity          Mid Cap          7
10   Equity        Small Cap          6
11   Equity            Value          1

## 3. AMFI code validation — fund_master vs nav_history

In [ ]:
fm_codes=set(fm['amfi_code']); nav_codes=set(d['nav_history']['amfi_code'])
print('fund_master codes :', len(fm_codes))
print('nav_history codes :', len(nav_codes))
print('missing in nav    :', len(fm_codes-nav_codes))
print('orphans in nav    :', len(nav_codes-fm_codes))
print('coverage          : {:.1%}'.format(1-len(fm_codes-nav_codes)/len(fm_codes)))

fund_master codes : 40
nav_history codes : 40
missing in nav    : 0
orphans in nav    : 0
coverage          : 100.0%


## 4. Live NAV snapshot (fetched via mfapi.in)

The AMFI codes in the brief are mislabelled — the table shows the actual schemes resolved from the API.

In [ ]:
nav_meta = pd.read_csv(RAW/'nav_meta.csv')
nav_meta[['scheme_code','requested_label','actual_scheme_name','records','nav_start','nav_end','latest_nav']]

   scheme_code                  requested_label                                                     actual_scheme_name  records   nav_start     nav_end  latest_nav
0       125497  HDFC Top 100 Direct (requested)                              SBI Small Cap Fund - Direct Plan - Growth     1745  2019-05-13  2026-05-31    193.6836
1       119551         SBI Bluechip (requested)         Aditya Birla Sun Life Banking & PSU Debt Fund  - DIRECT - IDCW     1757  2019-02-22  2026-06-01    104.7025
2       120503       ICICI Bluechip (requested)                 Axis ELSS Tax Saver Fund - Direct Plan - Growth Option     1753  2019-04-30  2026-06-01    103.0948
3       118632     Nippon Large Cap (requested)  Nippon India Large Cap Fund - Direct Plan Growth Plan - Growth Option        1  2026-05-26  2026-05-26     99.4356
4       119092        Axis Bluechip (requested)                   HDFC Money Market Fund - Growth Option - Direct Plan        1  2026-06-01  2026-06-01   6156.7532
5       120841  